
# ML 06 · Claim-frequency forecast, registered in Unity Catalog, joined with Lakebase team status

The daily watch is rules (is the loss ratio above threshold?) plus Genie (why?). This adds a **trained model**
that asks the forward-looking question: *will claim frequency keep rising next week?*

| Step | What |
|---|---|
| Features | per line × region and week: claim counts and frequency (claims per EUR 1m earned) over the last 1, 2, 4 and 13 weeks, earned premium, seasonality |
| Target | claim count in the **next 7 days** |
| Model | gradient-boosted trees (scikit-learn), time-based split (last 8 weeks held out) |
| Baseline | naive: next week = last week. The model must beat it to be promoted |
| Registry | MLflow → Unity Catalog model `lrc_frequency_forecast`, alias `champion` |
| Output | `lr_frequency_forecast` per segment, and `lr_forecast_escalations`: rising forecast **and** the team action in Lakebase still OPEN |

In [ ]:
%pip install -q "mlflow>=2.16" "scikit-learn>=1.4"

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [ ]:
dbutils.library.restartPython()

In [ ]:
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("schema", "loss_ratio_coworker")
CATALOG = dbutils.widgets.get("catalog") or spark.sql("SELECT current_catalog()").first()[0]
SCHEMA = dbutils.widgets.get("schema")
FQ = f"`{CATALOG}`.`{SCHEMA}`"
MODEL = f"{CATALOG}.{SCHEMA}.lrc_frequency_forecast"

## 1 · Weekly features from the governed loss run

In [ ]:
import numpy as np
import pandas as pd

daily = spark.sql(f"""
  SELECT as_of_date, line_of_business, region, SUM(claim_count) claims, SUM(earned_premium) premium
  FROM {FQ}.gold_daily_loss_run GROUP BY ALL""").toPandas()
daily["as_of_date"] = pd.to_datetime(daily["as_of_date"])
daily = daily.sort_values(["line_of_business", "region", "as_of_date"])

rows = []
for (lob, reg), g in daily.groupby(["line_of_business", "region"]):
    g = g.set_index("as_of_date").asfreq("D", fill_value=0)
    c, p = g["claims"].astype(float), g["premium"].astype(float)
    f = pd.DataFrame({
        "c7": c.rolling(7).sum(), "c14": c.rolling(14).sum(), "c28": c.rolling(28).sum(), "c91": c.rolling(91).sum(),
        "p7": p.rolling(7).sum(), "p28": p.rolling(28).sum(),
    })
    f["freq7"] = f.c7 / (f.p7 / 1e6)
    f["freq28"] = f.c28 / (f.p28 / 1e6)
    f["trend_7_vs_28"] = f.c7 / (f.c28 / 4).replace(0, np.nan)
    f["woy_sin"] = np.sin(2 * np.pi * f.index.isocalendar().week.astype(int) / 52)
    f["woy_cos"] = np.cos(2 * np.pi * f.index.isocalendar().week.astype(int) / 52)
    f["target_next7"] = c[::-1].rolling(7).sum()[::-1].shift(-1)      # claims in the next 7 days
    f["line_of_business"], f["region"] = lob, reg
    rows.append(f.iloc[::7])                                            # one row per week
data = pd.concat(rows).reset_index().rename(columns={"index": "as_of_date"})
data = data.dropna(subset=["c91", "trend_7_vs_28"])
seg = pd.get_dummies(data[["line_of_business", "region"]], dtype=float)
FEATURES = ["c7", "c14", "c28", "c91", "p7", "p28", "freq7", "freq28", "trend_7_vs_28", "woy_sin", "woy_cos"] + list(seg.columns)
X_all = pd.concat([data[FEATURES[:11]], seg], axis=1)
print(f"{len(data):,} weekly rows · {data[['line_of_business','region']].drop_duplicates().shape[0]} segments · {len(FEATURES)} features")

1,700 weekly rows · 20 segments · 20 features


## 2 · Train with a time-based hold-out and compare with the naive baseline

In [ ]:
import mlflow
from mlflow.models import infer_signature
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

labelled = data["target_next7"].notna()
cutoff = data.loc[labelled, "as_of_date"].max() - pd.Timedelta(weeks=8)
train = labelled & (data.as_of_date <= cutoff)
test = labelled & (data.as_of_date > cutoff)
X_tr, y_tr, X_te, y_te = X_all[train], data.loc[train, "target_next7"], X_all[test], data.loc[test, "target_next7"]

mlflow.set_registry_uri("databricks-uc")
me = spark.sql("SELECT current_user()").first()[0]
mlflow.set_experiment(f"/Users/<user>/lrc_frequency_forecast")
with mlflow.start_run(run_name="hgb_next7_claims") as run:
    model = HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05, max_leaf_nodes=15, random_state=7)
    model.fit(X_tr, y_tr)
    pred = model.predict(X_te)
    mae_model = mean_absolute_error(y_te, pred)
    mae_naive = mean_absolute_error(y_te, X_te["c7"])
    improvement = 1 - mae_model / mae_naive
    mlflow.log_params({"model": "HistGradientBoostingRegressor", "max_iter": 300, "learning_rate": 0.05, "holdout_weeks": 8})
    mlflow.log_metrics({"mae_model": mae_model, "mae_naive": mae_naive, "improvement_vs_naive": improvement,
                        "train_rows": len(X_tr), "test_rows": len(X_te)})
    info = mlflow.sklearn.log_model(model, "model", signature=infer_signature(X_tr, model.predict(X_tr.head(5))),
                                    input_example=X_tr.head(3), registered_model_name=MODEL)
print(f"hold-out MAE: model {mae_model:.2f} vs naive {mae_naive:.2f} claims/week → {improvement:+.1%} better than naive")

If you are using MLflow Tracing, you can migrate your traces to Unity Catalog for unlimited storage, fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/migrate-traces-to-uc
2026/11/16 05:15:10 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: <workspace-url>
Registered model '<catalog>.loss_ratio_coworker.lrc_frequency_forecast' already exists. Creating a new version of this model...


🔗 Created version '2' of model '<catalog>.loss_ratio_coworker.lrc_frequency_forecast': <workspace-url>


hold-out MAE: model 5.44 vs naive 6.84 claims/week → +20.4% better than naive


## 3 · Promote only if it beats the baseline

In [ ]:
from mlflow import MlflowClient
client = MlflowClient(registry_uri="databricks-uc")
version = max(int(v.version) for v in client.search_model_versions(f"name='{MODEL}'"))
if improvement > 0:
    client.set_registered_model_alias(MODEL, "champion", version)
    print(f"{MODEL} v{version} → alias champion")
else:
    print(f"{MODEL} v{version} not promoted: it does not beat the naive baseline")

<catalog>.loss_ratio_coworker.lrc_frequency_forecast v2 → alias champion


## 4 · Score the latest week with the champion and combine with Lakebase team status

In [ ]:
champion = mlflow.sklearn.load_model(f"models:/{MODEL}@champion")
latest = data.groupby(["line_of_business", "region"]).tail(1)
Xl = X_all.loc[latest.index]
out = latest[["as_of_date", "line_of_business", "region", "c7", "c28"]].copy()
out["forecast_next7_claims"] = np.round(champion.predict(Xl), 1)
out["weekly_avg_last4"] = out.c28 / 4
out["forecast_vs_4wk_avg_pct"] = np.round(100 * (out.forecast_next7_claims / out.weekly_avg_last4 - 1), 1)
out["frequency_outlook"] = np.where(out.forecast_vs_4wk_avg_pct > 15, "RISING", np.where(out.forecast_vs_4wk_avg_pct < -15, "FALLING", "STABLE"))
out["model_version"] = int(version)
out = out.rename(columns={"c7": "claims_last7"}).drop(columns=["c28"])
spark.createDataFrame(out).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{FQ}.lr_frequency_forecast")
spark.sql(f"COMMENT ON TABLE {FQ}.lr_frequency_forecast IS 'Next-7-day claim-count forecast per segment from UC model lrc_frequency_forecast@champion.'")
display(spark.table(f"{FQ}.lr_frequency_forecast").orderBy("forecast_vs_4wk_avg_pct", ascending=False))

| as_of_date | line_of_business | region | claims_last7 | forecast_next7_claims | weekly_avg_last4 | forecast_vs_4wk_avg_pct | frequency_outlook | model_version |
|---|---|---|---|---|---|---|---|---|
| 2026-11-11T00:00:00.000Z | Marine | Benelux | 8.0 | 10.8 | 8.25 | 30.9 | RISING | 2 |
| 2026-11-11T00:00:00.000Z | Liability | UK & Ireland | 26.0 | 29.7 | 24.25 | 22.5 | RISING | 2 |
| 2026-11-11T00:00:00.000Z | Property | Southern Europe | 24.0 | 37.1 | 32.0 | 15.9 | RISING | 2 |
| 2026-11-11T00:00:00.000Z | Marine | France | 16.0 | 18.6 | 16.25 | 14.5 | STABLE | 2 |
| 2026-11-11T00:00:00.000Z | Marine | Southern Europe | 12.0 | 15.6 | 14.0 | 11.4 | STABLE | 2 |
| 2026-11-11T00:00:00.000Z | Marine | DACH | 23.0 | 22.3 | 21.0 | 6.2 | STABLE | 2 |
| 2026-11-11T00:00:00.000Z | Property | UK & Ireland | 46.0 | 48.4 | 46.75 | 3.5 | STABLE | 2 |
| 2026-11-11T00:00:00.000Z | Liability | Benelux | 20.0 | 13.9 | 13.5 | 3.0 | STABLE | 2 |
| 2026-11-11T00:00:00.000Z | Motor | France | 85.0 | 85.1 | 83.0 | 2.5 | STABLE | 2 |
| 2026-11-11T00:00:00.000Z | Motor | Benelux | 56.0 | 53.3 | 52.25 | 2.0 | STABLE | 2 |
| 2026-11-11T00:00:00.000Z | Property | France | 39.0 | 38.2 | 38.25 | -0.1 | STABLE | 2 |
| 2026-11-11T00:00:00.000Z | Property | Benelux | 22.0 | 23.1 | 23.5 | -1.7 | STABLE | 2 |
| 2026-11-11T00:00:00.000Z | Motor | Southern Europe | 87.0 | 75.8 | 78.75 | -3.7 | STABLE | 2 |
| 2026-11-11T00:00:00.000Z | Motor | DACH | 98.0 | 92.1 | 96.0 | -4.1 | STABLE | 2 |
| 2026-11-11T00:00:00.000Z | Liability | Southern Europe | 17.0 | 20.1 | 21.0 | -4.3 | STABLE | 2 |
| 2026-11-11T00:00:00.000Z | Property | DACH | 81.0 | 47.9 | 50.25 | -4.7 | STABLE | 2 |
| 2026-11-11T00:00:00.000Z | Liability | DACH | 27.0 | 24.9 | 26.75 | -6.9 | STABLE | 2 |
| 2026-11-11T00:00:00.000Z | Liability | France | 19.0 | 21.6 | 23.75 | -9.1 | STABLE | 2 |
| 2026-11-11T00:00:00.000Z | Marine | UK & Ireland | 21.0 | 21.5 | 25.0 | -14.0 | STABLE | 2 |
| 2026-11-11T00:00:00.000Z | Motor | UK & Ireland | 149.0 | 109.1 | 127.25 | -14.3 | STABLE | 2 |


**Escalation rule:** the model expects frequency to keep rising **and** the owning team's action in Lakebase
(`lr_action_status`, snapshotted from `ops.lr_actions`) is still OPEN. That is the list a CUO wants on Monday.

In [ ]:
esc = spark.sql(f"""
SELECT f.line_of_business, f.region, f.claims_last7, f.forecast_next7_claims, f.forecast_vs_4wk_avg_pct,
       a.team, a.priority, a.status AS lakebase_status, a.action
FROM {FQ}.lr_frequency_forecast f
JOIN {FQ}.lr_action_status a USING (line_of_business, region)
WHERE f.frequency_outlook = 'RISING' AND a.status = 'OPEN'
""")
esc.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{FQ}.lr_forecast_escalations")
spark.sql(f"COMMENT ON TABLE {FQ}.lr_forecast_escalations IS 'Segments where the frequency forecast is rising and the Lakebase team action is still OPEN.'")
display(esc)

| line_of_business | region | claims_last7 | forecast_next7_claims | forecast_vs_4wk_avg_pct | team | priority | lakebase_status | action |
|---|---|---|---|---|---|---|---|---|
| Liability | UK & Ireland | 26.0 | 29.7 | 22.5 | Underwriting | P2 | OPEN | Monitor. No action unless the 30-day loss ratio also breaches. |